# Topic 3: Built-in Functions

> **Phase 3 → Week 4 → Topic 3**
>
> Prerequisites: Topic 1 & 2

---

## Why Built-in Functions Over Python UDFs

```
Python UDF:              Built-in Function:
@udf                     F.upper(col)
def my_upper(s):    vs
    return s.upper()

- Python process        - Runs in JVM (Tungsten)
- Serialization cost    - No serialization
- Catalyst can't        - Catalyst optimizes
  optimize              - 10-100x faster
- Slower                - Always prefer these
```

**Rule: Always use `pyspark.sql.functions` over Python UDFs whenever possible.**

---

## Interview Cheat Sheet

**Q: Why are Spark built-in functions faster than Python UDFs?**
> Built-in functions execute natively in the JVM via Tungsten — no Python-JVM serialization, and Catalyst can inspect and optimize them. Python UDFs require serializing each row from JVM to Python process, executing, then serializing back — row by row overhead adds up to 10-100x slower.

**Q: What's the difference between when() and coalesce() for conditional logic?**
> `when(condition, value).otherwise(default)` is general conditional logic — like an IF/ELSE. `coalesce(col1, col2, ...)` returns the FIRST non-null value from its arguments — it's specifically for null fallback/default values, not general conditions.

**Q: How do you handle date arithmetic in Spark?**
> Use `F.datediff(end, start)` for days between dates, `F.months_between(end, start)` for months, `F.date_add(col, n)` / `F.date_sub(col, n)` to add/subtract days. Always use `F.to_date(col, format)` or `F.to_timestamp()` to parse string dates first.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week4 - Built-in Functions") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
print("Data loaded")

## Part 1: String Functions

In [ ]:
string_df = spark.createDataFrame([
    ("Alice Sharma",   "alice@example.com",   "  hello world  "),
    ("Bob Chen",       "bob.chen@gmail.com",   "  SPARK is FUN  "),
    ("Carol Williams", "carol@company.co.uk",  "data engineering"),
], ["name", "email", "notes"])

string_ops = string_df.select(
    "name",
    F.upper("name").alias("upper_name"),
    F.lower("name").alias("lower_name"),
    F.length("name").alias("name_len"),
    F.trim("notes").alias("trimmed"),
    F.ltrim("notes").alias("ltrimmed"),
    F.split("name", " ")[0].alias("first_name"),
    F.split("name", " ")[1].alias("last_name"),
    F.substring("email", 1, 5).alias("email_prefix"),
    F.regexp_extract("email", r"@(.+)", 1).alias("email_domain"),
    F.regexp_replace("notes", "  ", " ").alias("single_space"),
)

print("String functions:")
string_ops.show(truncate=False)

In [ ]:
# concat, concat_ws, format_string, lpad, rpad
concat_df = customers.select(
    F.concat(F.col("name"), F.lit(" ("), F.col("country"), F.lit(")")).alias("name_country"),
    F.concat_ws(" | ", "name", "city", "country").alias("pipe_separated"),
    F.format_string("Customer %s from %s", "name", "country").alias("formatted"),
    F.lpad(F.col("customer_id").cast("string"), 4, "0").alias("padded_id"),
    F.initcap(F.lower("country")).alias("title_case"),
)

print("Concat, format, pad functions:")
concat_df.show(truncate=False)

In [ ]:
# contains, startswith, endswith, like, rlike
filter_df = customers.filter(
    F.col("name").contains("a")   # name contains lowercase 'a'
).select("name", "country")

print("Names containing 'a':")
filter_df.show()

# Pattern matching
pattern_df = customers.select(
    "name",
    F.col("name").startswith("A").alias("starts_A"),
    F.col("name").endswith("n").alias("ends_n"),
    F.col("name").like("%ar%").alias("sql_like"),
    F.col("name").rlike("^[A-E]").alias("regex_A_to_E"),
)
print("Pattern matching:")
pattern_df.show()

## Part 2: Date & Time Functions

In [ ]:
date_df = orders.select("order_id", "order_date", "amount")

date_ops = date_df.withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd")) \
                  .withColumn("year",        F.year("order_date")) \
                  .withColumn("month",       F.month("order_date")) \
                  .withColumn("day",         F.dayofmonth("order_date")) \
                  .withColumn("day_of_week", F.dayofweek("order_date")) \
                  .withColumn("week_of_yr",  F.weekofyear("order_date")) \
                  .withColumn("quarter",     F.quarter("order_date")) \
                  .withColumn("plus_30days", F.date_add("order_date", 30)) \
                  .withColumn("today",       F.current_date()) \
                  .withColumn("days_ago",    F.datediff(F.current_date(), "order_date"))

print("Date extraction and arithmetic:")
date_ops.show(truncate=False)

In [ ]:
# Timestamp functions
ts_df = spark.createDataFrame([
    ("2023-06-15 14:30:00",),
    ("2023-06-16 09:15:30",),
    ("2023-12-31 23:59:59",),
], ["ts_string"])

ts_ops = ts_df.withColumn("ts",       F.to_timestamp("ts_string", "yyyy-MM-dd HH:mm:ss")) \
              .withColumn("hour",     F.hour("ts")) \
              .withColumn("minute",   F.minute("ts")) \
              .withColumn("date",     F.to_date("ts")) \
              .withColumn("unix_ts",  F.unix_timestamp("ts")) \
              .withColumn("truncate_hr", F.date_trunc("hour", "ts"))  # round down to hour

print("Timestamp functions:")
ts_ops.show(truncate=False)

In [ ]:
# months_between + date formatting
signup_analysis = customers.withColumn(
    "signup_date", F.to_date("signup_date", "yyyy-MM-dd")
).withColumn(
    "months_since_signup", F.round(F.months_between(F.current_date(), "signup_date"), 1)
).withColumn(
    "signup_formatted",    F.date_format("signup_date", "dd MMM yyyy")
).select("name", "signup_date", "signup_formatted", "months_since_signup")

print("Months since signup:")
signup_analysis.show()

## Part 3: Math Functions

In [ ]:
math_df = spark.createDataFrame([
    (1,  -15.7,  2.5,  100.0),
    (2,   23.4,  0.0,  200.0),
    (3,  -0.3,   9.0,   50.0),
    (4,   87.6, -3.0,  150.0),
], ["id", "val", "divisor", "price"])

math_ops = math_df.select(
    "id",
    F.abs("val").alias("abs_val"),
    F.round("val", 1).alias("rounded_1dp"),
    F.ceil("val").alias("ceiling"),
    F.floor("val").alias("floor"),
    F.sqrt("price").alias("sqrt_price"),
    F.pow("price", 2).alias("price_squared"),
    F.log("price").alias("log_price"),
    F.log(10.0, "price").alias("log10_price"),
    F.when(F.col("divisor") != 0, F.col("price") / F.col("divisor")).alias("safe_divide"),
)

print("Math functions:")
math_ops.show()

## Part 4: Conditional Functions — when(), otherwise(), coalesce()

In [ ]:
# when().when().otherwise() — like IF/ELSEIF/ELSE or SQL CASE WHEN
orders_with_tier = orders.withColumn(
    "order_tier",
    F.when(F.col("amount") >= 1000, "High")
     .when(F.col("amount") >= 200,  "Medium")
     .when(F.col("amount") >= 50,   "Low")
     .otherwise("Micro")
)

print("Order tiers using when/otherwise:")
orders_with_tier.select("order_id", "amount", "order_tier").show()

print("Count per tier:")
orders_with_tier.groupBy("order_tier").count().show()

In [ ]:
# coalesce() — first non-null from list of columns
# Great for providing fallback values across columns

fallback_df = spark.createDataFrame([
    (1, None,    None,    "default_A"),
    (2, "val_B", None,    "default_B"),
    (3, None,    "val_C2", "default_C"),
    (4, "val_D", "val_D2", "default_D"),
], ["id", "col1", "col2", "col3"])

fallback_df.withColumn(
    "first_non_null", F.coalesce("col1", "col2", "col3")
).show()

# nvl (alias for coalesce with two args) — SQL compatibility
spark_df = customers.withColumn(
    "tier_or_unknown", F.coalesce(F.col("tier"), F.lit("Unknown"))
)
print("Coalesce for null fallback:")
spark_df.select("name", "tier", "tier_or_unknown").show(5)

In [ ]:
# Conditional aggregation — count/sum only rows meeting a condition
# Pattern: F.sum(F.when(condition, 1).otherwise(0))

order_summary = orders.groupBy("customer_id").agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered_count"),
    F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_count"),
    F.sum(F.when(F.col("status") == "delivered", F.col("amount")).otherwise(0)).alias("delivered_revenue"),
    F.round(F.avg("amount"), 2).alias("avg_order_value")
)

print("Conditional aggregation (pivot-style):")
order_summary.show()

## Part 5: Aggregate Functions Reference

In [ ]:
# Common aggregate functions in one place
agg_demo = orders.agg(
    F.count("order_id").alias("total_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order"),
    F.min("amount").alias("min_order"),
    F.max("amount").alias("max_order"),
    F.stddev("amount").alias("stddev_amount"),
    F.variance("amount").alias("variance"),
    F.first("order_date").alias("first_order_date"),
    F.last("order_date").alias("last_order_date"),
    F.collect_list("status").alias("all_statuses"),
    F.collect_set("status").alias("unique_statuses"),
)

print("Full aggregate summary of orders:")
agg_demo.show(truncate=False)

## Exercises

1. From the `customers` table, extract the first name and last name into separate columns using `split()`.
2. For each order, calculate how many days ago it was placed (from today's date).
3. Using `when().otherwise()`, classify customers into: `New` (signup < 1 year ago), `Regular` (1-2 years), `Veteran` (2+ years).
4. Find the total amount for delivered orders vs non-delivered orders using conditional aggregation.
5. Build a full customer summary: name, total orders, total spend, avg order value, last order date.

In [ ]:
# Exercise 1: Extract first/last name
customers.select(
    F.split("name", " ")[0].alias("first_name"),
    F.split("name", " ")[1].alias("last_name"),
    "country"
).show()

In [ ]:
# Exercise 3: Customer vintage classification
from pyspark.sql.window import Window

customers.withColumn(
    "signup_date", F.to_date("signup_date")
).withColumn(
    "months_ago", F.months_between(F.current_date(), "signup_date")
).withColumn(
    "segment",
    F.when(F.col("months_ago") < 12,  "New")
     .when(F.col("months_ago") < 24,  "Regular")
     .otherwise("Veteran")
).select("name", "signup_date", "months_ago", "segment").show()